# 《后分析篇》I.21：否定推导与混合路径终止

原创教学实验，不是原典全文翻译或一般无限定理的机器证明。只用 Python 标准库。请在此目录从头运行。

## 1. 先固定谓词方向
All(P,S) 表示所有 S 都是 P；E(P,S) 表示二者无交；O(P,S) 表示至少一个 S 不是 P。空主项不提供存在见证。

In [1]:
import json
from model import All, E, O, semantic_results, ranking_results, switching_example, measure, successors
print(json.dumps({'All(3,1)':All(3,1),'All(1,3)':All(1,3),'E(1,0)':E(1,0),'O(1,0)':O(1,0)},ensure_ascii=False,indent=2))


{
  "All(3,1)": true,
  "All(1,3)": false,
  "E(1,0)": true,
  "O(1,0)": false
}


## 2. 穷举有限三项集合
整数位掩码表示集合，论域大小逐一取 0 至 4。每个大小内部枚举所有 A/B/C；跨论域大小的同名位图是不同实验实例。

In [2]:
semantic = semantic_results()
print(json.dumps(semantic,ensure_ascii=False,indent=2))


{
  "universe_sizes": [
    0,
    1,
    2,
    3,
    4
  ],
  "assignments": 4681,
  "rules": {
    "first_EAE": {
      "premise_cases": 341,
      "counterexamples": 0
    },
    "second_AEE": {
      "premise_cases": 341,
      "counterexamples": 0
    },
    "third_AOO": {
      "premise_cases": 774,
      "counterexamples": 0
    }
  },
  "E_to_O_with_nonempty_subject": {
    "premise_cases": 90,
    "counterexamples": 0
  },
  "empty_subject_example": {
    "P": 1,
    "S": 0,
    "E": true,
    "O": false
  },
  "O_not_E_example": {
    "P": 1,
    "S": 3,
    "O": true,
    "E": false
  },
  "third_figure_conclusion_not_universal": {
    "A": 3,
    "B": 1,
    "C": 2,
    "premises": true,
    "O_C_A": true,
    "E_C_A": false
  }
}


## 3. 逆向展开的统一测度
E目标的两端严格向上；O目标谓词向上或主项向下。O目标可以借非空证据转去证明更强的E前提，这不是从O推出E。该骨架只验证进展，不认证前提真实、因果相关或所有原典规则。

In [3]:
rank = ranking_results(4)
print(json.dumps(rank,ensure_ascii=False,indent=2))
start=('O',0,4)
print('initial measure:',measure(start,4))
print('backward moves with nonempty evidence:',successors(start,4,True))


{
  "height": 4,
  "states": 50,
  "edges": 225,
  "strict_decrease_violations": 0,
  "max_initial_measure": 17,
  "max_backward_steps": 13,
  "O_to_E_requires_nonempty_evidence": true,
  "same_state_or_equivalence_moves_included": false
}
initial measure: 17
backward moves with nonempty evidence: [('raise_predicate', ('O', 1, 4)), ('raise_predicate', ('O', 2, 4)), ('raise_predicate', ('O', 3, 4)), ('raise_predicate', ('O', 4, 4)), ('lower_subject', ('O', 0, 0)), ('lower_subject', ('O', 0, 1)), ('lower_subject', ('O', 0, 2)), ('lower_subject', ('O', 0, 3)), ('existential_weakening_backwards', ('E', 0, 4))]


## 4. 为什么有限种方法还不够
R只有u到v，S只有v到u。分别重复都终止，但轮换可循环。有限种格的数量本身不是统一终止证据。

In [4]:
print(json.dumps(switching_example(),ensure_ascii=False,indent=2))


{
  "R": [
    [
      "u",
      "v"
    ]
  ],
  "S": [
    [
      "v",
      "u"
    ]
  ],
  "R_has_cycle": false,
  "S_has_cycle": false,
  "union_has_cycle": true,
  "alternating_prefix": [
    "u",
    "v",
    "u",
    "v",
    "u"
  ]
}


## 复盘
把H改为其他小整数，比较最长路径与测度上界。删去非空证据时，O→E逆向边为何消失？若允许同项反复或等价项互换，严格下降在哪一步失效？

答案：测度只保证已声明的严格进展；非空证据是全否蕴涵特否的附加条件；同项/等价互换不降低秩，因此不属于该终止定理。有限测试支持代码的一致性，不证明科学谓述必有上下终点。